In [1]:
import sys
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from config.config import RAW_DATA_FILE, PROCESSED_DATA_FILE
from src.data.loader import load_raw_data
from src.cleaning.cleaner import clean_dataset, validate_logical_consistency

df_raw = load_raw_data(RAW_DATA_FILE)
df_raw.shape

(25000, 15)

## Check for logical inconsistencies before cleaning 

In [3]:
inconsistent = validate_logical_consistency(df_raw)
print(f"Inconsistent rows found: {inconsistent.shape[0]}")
inconsistent.head()

Inconsistent rows found: 0


,delivery_id,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,delivery_time_hours,expected_time_hours,delayed,delivery_status,delivery_rating,delivery_cost


## Run the cleaning pipeline

In [4]:
df_clean = clean_dataset(df_raw)
df_clean.shape

(25000, 16)

## Verify the decoded time columns

In [5]:
df_clean[["delivery_time_hours_clean", "expected_time_hours_clean"]].describe()

,delivery_time_hours_clean,expected_time_hours_clean
count,25000.000000,25000.000000
mean,6.248040,13.107680
std,3.140935,7.559024
min,0.000000,2.000000
25%,4.000000,8.000000
50%,6.000000,8.000000
75%,8.000000,16.000000
max,19.000000,24.000000


## Confirm no data was silently lost

In [6]:
print("Raw rows:", df_raw.shape[0])
print("Clean rows:", df_clean.shape[0])
assert df_raw.shape[0] == df_clean.shape[0], "Row count changed unexpectedly during cleaning!"

Raw rows: 25000
Clean rows: 25000


## Spot-check a few rows before/after

In [7]:
df_clean.head(5)

,row_id,delivery_id,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,delayed,delivery_status,delivery_rating,delivery_cost,delivery_time_hours_clean,expected_time_hours_clean
0,1,250.99,delhivery,automobile parts,bike,same day,west,clear,297.0,46.96,no,delivered,3,1632.7206,8.0,8.0
1,2,250.99,xpressbees,cosmetics,ev van,express,central,cold,89.6,47.39,no,delivered,5,640.1700,2.0,3.0
2,3,250.99,shadowfax,groceries,truck,two day,east,rainy,273.5,26.89,no,delivered,4,1448.1700,10.0,16.0
3,4,250.99,dhl,electronics,ev van,same day,east,cold,269.7,12.69,no,delivered,3,1486.5700,6.0,8.0
4,5,250.99,dhl,clothing,van,two day,north,foggy,256.7,37.02,no,delivered,4,1394.5600,9.0,16.0


## Outlier review (decision, not blind removal)

In [8]:
# Reviewing IQR-flagged outliers from Data Understanding.
# Decision: distance_km, package_weight_kg, delivery_cost outliers are
# kept -- they fall within physically plausible ranges (e.g. max distance
# 297km, max weight ~49.5kg) and likely represent genuine long-haul or
# heavy-item deliveries, which are analytically relevant, not data errors.
# No rows are removed on this basis.

for col in ["distance_km", "package_weight_kg", "delivery_cost"]:
    q1, q3 = df_clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    print(f"{col}: plausible range check -> min={df_clean[col].min()}, "
          f"max={df_clean[col].max()}, IQR bounds=({lower:.1f}, {upper:.1f})")

distance_km: plausible range check -> min=3.6, max=297.1, IQR bounds=(-147.6, 448.4)
package_weight_kg: plausible range check -> min=0.67, max=49.52, IQR bounds=(-24.8, 75.1)
delivery_cost: plausible range check -> min=95.6674, max=1632.7206, IQR bounds=(-629.9, 2358.6)


## Save the cleaned dataset

In [9]:
PROCESSED_DATA_FILE.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(PROCESSED_DATA_FILE, index=False)
print(f"Saved cleaned dataset to: {PROCESSED_DATA_FILE}")

Saved cleaned dataset to: E:\Python projects\last-mile-delivery-analysis\data\processed\delivery_logistics_clean.csv


## Cleaning Decisions Log

1. **row_id added** -- `delivery_id` cannot be trusted as a unique key
   (500 colliding rows due to float rounding). A guaranteed-unique
   `row_id` was added instead. `delivery_id` itself was kept in the
   dataset for reference, not dropped, since we have no evidence it's
   meaningless -- only that it's not unique.

2. **Time columns decoded** -- `delivery_time_hours` and
   `expected_time_hours` were malformed timestamp strings. Decoded into
   `delivery_time_hours_clean` and `expected_time_hours_clean` using the
   logic validated in Data Understanding (correlation with distance and
   cost). Original malformed columns dropped after decoding to avoid
   duplicate/confusing fields downstream.

3. **No missing values, no full-row duplicates** -- confirmed again post
   cleaning, no action needed.

4. **`delayed` / `delivery_status` consistency** -- checked, zero
   inconsistent rows found (see the check above). No correction needed.

5. **Outliers in distance, weight, cost** -- reviewed, not removed. All
   values fall within physically plausible real-world ranges for a
   delivery business (e.g. heavy furniture items, long cross-region
   trips). Removing them would discard genuine business signal.

6. **Categorical columns** -- no standardization needed; consistent category labels across all 8 categorical fields.
